In [ ]:
# 第9周-Day2：SkillRelease — 为什么它是唯一可部署单元？
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

plt.title("中文 English 123")
plt.xlabel("时间 Time 2026")
plt.ylabel("数值 Value -10")
plt.plot([1, 2, 3], [10, 20, 15])
plt.show()

## 📅 Week 9 - Day 2 | 2026-07-28

### SkillRelease：为什么它是唯一可部署单元？

| 项目 | 内容 |
|---|---|
| **本周主题** | Domain Deep Dive — 拆对象，理解为什么存在 |
| **今日对象** | SkillRelease（Domain Model §7.4 SC-04）|
| **ADR 依据** | ADR-001 §4.5, ADR-003 v1.2, ADR-005 D-5, ADR-006 D-1 |
| **核心问题** | 为什么 SkillRelease 是唯一可部署单元，而不是 Blueprint/Capability/Workflow？ |

### 学习进度
■■■■■■■■■■■■■■■■■■■■□□□□□□□□□□□□□□□□□□□
W1-W7 ✅  W8 ✅  W9 🔵 进行中（Day2/7）  W10-W18 待解锁

## ❓ 今日核心问题

为什么 SkillRelease 是唯一可部署单元，而不是 Blueprint、Capability 或 Workflow？

因为这三个都只是"零件"，不是"产品"。SkillRelease 是唯一经过完整制品供应链的输出物：

1. ✅ **确定性构建**（从 Blueprint → IR → 打包）
2. ✅ **精确依赖锁**（不允许 latest，所有依赖必须 digest 锁定）
3. ✅ **Release Gate 通过证据**（QA 测试、安全扫描）
4. ✅ **数字签名**（防篡改）
5. ✅ **不可变生命周期**（发布后不能修改，只能新建版本）

## 🗣 人话解释（Jason 26年 ERP 经验）

想象你给客户发 ERP 安装包。你不会发源代码（=Blueprint），不会发单个模块（=Capability），也不会发开发中的工作流配置（=WorkflowSpec）。

你需要：编译（Build→IR）→ 打包含依赖锁（Packaging）→ QA测试（Release Gate）→ 数字签名（Signature）→ 发布到渠道（ReleaseChannel）。

SkillRelease 就是那个"经过签名认证的安装包"。客户（Agent Host）只能通过这个安装包使用你的产品。

为什么不允许直接暴露 Capability/Workflow API？因为那样就绕过了制品供应链的质量管控。

In [ ]:
# SkillRelease 在供应链中的位置
fig, ax = plt.subplots(figsize=(14, 4))

# Supply Chain Layer
chain = ['Blueprint', 'Build', 'IR', 'SkillRelease', 'Deployment', 'Execute']
colors = ['#4ECDC4', '#4ECDC4', '#4ECDC4', '#FF6B6B', '#45B7D1', '#45B7D1']
layers = ['Supply Chain', 'Supply Chain', 'Supply Chain', '制品终点', 'Runtime', 'Runtime']

bars = ax.bar(range(len(chain)), [1]*len(chain), color=colors, edgecolor='white', linewidth=2)

for i, (name, layer) in enumerate(zip(chain, layers)):
    ax.text(i, 0.5, name, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    ax.text(i, -0.15, layer, ha='center', va='top', fontsize=8, color='#666')

# Arrow pointing to SkillRelease
ax.annotate('★ 唯一可部署单元', xy=(3, 1.1), fontsize=12, fontweight='bold',
            ha='center', color='#FF6B6B',
            arrowprops=dict(arrowstyle='->', color='#FF6B6B', lw=2))

# Boundary line
ax.axvline(x=2.5, color='#999', linestyle='--', alpha=0.5)
ax.axvline(x=3.5, color='#999', linestyle='--', alpha=0.5)
ax.text(1.5, 1.3, 'Supply Chain Layer', ha='center', fontsize=9, color='#999')
ax.text(4.5, 1.3, 'Runtime Layer', ha='center', fontsize=9, color='#999')

ax.set_xlim(-0.5, len(chain)-0.5)
ax.set_ylim(-0.3, 1.5)
ax.axis('off')
ax.set_title('SkillRelease：供应链终点 = Runtime入口', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 📋 ADR 依据

### ADR-001 §4.5：P0 唯一消费单元
SkillRelease 是 P0 唯一对外消费与发布单元，不暴露 Capability/Workflow API。

### ADR-003 v1.2：冻结 5 个 HTTP 端点

| 端点 | 方法 | 用途 |
|------|------|------|
| `/v1/skill-releases` | GET | 发现可用 Skill |
| `/v1/skill-releases/{id}` | GET | 获取 Skill 描述 |
| `/invoke` | POST | 调用（同步200或HITL 202） |
| `/executions/{id}` | GET | 查询执行状态 |
| `/executions/{id}/respond` | POST | 人审响应 |

关键约束：
- **HC-4**: P0只读（effect_policy=read_only）
- **HC-5**: 唯一对外消费单元
- **存在性保护**：无权调用的skill返回404而非403

### ADR-005 D-5：WorkflowSpec 目标态退役
当前 W01-W09 binding 是过渡态，最终 SkillRelease v2 制品直接驱动 Runtime。

### ADR-006 D-1：语义锚点分离
DigitalEmployeeDefinition 是"谁"，SkillRelease 是"什么"，两者不合并。

In [ ]:
# ADR-003 五端点冻结可视化
fig, ax = plt.subplots(figsize=(12, 5))

endpoints = [
    ('GET\n/v1/skill-releases', '发现', '#4ECDC4'),
    ('GET\n/v1/skill-releases/{id}', '描述', '#4ECDC4'),
    ('POST\n/invoke', '调用', '#FF6B6B'),
    ('GET\n/executions/{id}', '查询', '#4ECDC4'),
    ('POST\n/executions/{id}/respond', '人审', '#FFD93D'),
]

for i, (ep, desc, color) in enumerate(endpoints):
    circle = plt.Circle((i*2.2+1, 2), 0.8, color=color, alpha=0.8)
    ax.add_patch(circle)
    ax.text(i*2.2+1, 2, ep, ha='center', va='center', fontsize=8, fontweight='bold', color='white' if color != '#FFD93D' else '#333')
    ax.text(i*2.2+1, 0.8, desc, ha='center', va='center', fontsize=10, color='#666')

# Constraints
ax.text(5.5, 3.5, 'HC-4: P0只读 (read_only)', fontsize=9, color='#FF6B6B', fontweight='bold')
ax.text(5.5, 3.1, 'HC-5: 唯一对外消费单元', fontsize=9, color='#FF6B6B', fontweight='bold')
ax.text(5.5, 2.7, '存在性保护: 404 ≠ 403', fontsize=9, color='#FF6B6B', fontweight='bold')

ax.set_xlim(-0.5, 11.5)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('ADR-003 v1.2: SkillRelease 5 端点冻结', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔍 代码验证（/root/langchat）

### canonical/router.py
确认 5 端点与 ADR-003 一一对应。`_public_descriptor()` 把内部 Descriptor 转为对外 PublicModel，隐藏实现细节。

### bindings/w01~w09.py
当前 9 个 binding 文件把 SkillRelease 绑到 WorkflowSpec——这就是 cutover 后要替换的部分。

### 数据模型
| 表 | 用途 |
|---|---|
| canonical_execution | 六态状态机（pending→running→succeeded/failed/cancelled/timeout）|
| review_assignee | HITL 人审分配 |
| approval | 发布审批流 |
| workflow_binding | 退役目标（cutover 后删除）|

## 🏢 商业地产映射

| LangChat 概念 | CRE 对应 |
|---|---|
| SkillRelease | "合同查询数字员工"的签名安装包 |
| skill_id | cre.lease.query.v1（稳定标识，类似MI模块编号）|
| effect_policy=read_only | 只查不改，不写回MI |
| human_review_gate | 敏感问题（租金/违约金）需人工审核 |
| ReleaseChannel | dev→UAT→production 环境晋升 |
| DeploymentRevision | 某次部署的完整冻结快照 |

**为什么不能一个大 SkillRelease 包含所有？** 因为权限隔离、独立版本管理、独立灰度发布。招商模块升级不应该影响物业模块。

In [ ]:
# 传统 ERP API vs LangChat SkillRelease 对比
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

data = [
    ['暴露方式', '前后端耦合 REST API', '封装为制品，隐藏内部实现'],
    ['版本管理', 'Git分支 + 变更日志', 'OCI制品 digest + 语义化版本'],
    ['依赖管理', 'pom.xml（允许范围版本）', '精确依赖锁（必须digest）'],
    ['质量管控', '人工流程', 'Release Gate 自动化'],
    ['回滚方式', '代码回滚 + DB迁移', '前向回滚（物化新Revision）'],
    ['权限模型', 'API网关层 ACL', 'effect_policy 内嵌 + 存在性保护'],
]

col_labels = ['维度', '传统 ERP API', 'LangChat SkillRelease']
table = ax.table(cellText=data, colLabels=col_labels, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

# Style header
for j in range(3):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(data)+1):
    for j in range(3):
        if j == 2:
            table[i, j].set_facecolor('#E8F8F5')
        elif j == 1:
            table[i, j].set_facecolor('#FDEDEC')

ax.set_title('传统 ERP API vs LangChat SkillRelease', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## Gap Analysis

| 目标态 | 当前代码 | Gap 等级 |
|---|---|---|
| SkillRelease 直接驱动 Runtime | 需要 WorkflowSpec binding 中间层 | P1（ADR-005 cutover 目标）|
| OCI 制品格式 | 当前是数据库记录 | P2（v2 目标）|
| 精确依赖锁（digest）| 当前用语义化版本 | P2 |
| Release Gate 自动化 | 当前手动审批 | P1 |
| 存在性保护（404 vs 403）| 代码已实现 | ✅ 无 Gap |
| 5 端点冻结 | 代码已对齐 | ✅ 无 Gap |

**最大风险**：WorkflowSpec binding（9个文件）是技术债，cutover 越晚技术债越大。

In [ ]:
# SkillRelease 生命周期 + canonical_execution 六态状态机
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: SkillRelease lifecycle
ax1 = axes[0]
stages = ['Blueprint\n(设计)', 'Build\n(编译)', 'IR\n(中间表示)', 'SkillRelease\n(签名制品)', 'Deployment\n(部署)']
stage_colors = ['#74B9FF', '#74B9FF', '#74B9FF', '#E17055', '#00B894']

for i, (stage, color) in enumerate(zip(stages, stage_colors)):
    rect = plt.Rectangle((i*2, 0), 1.5, 1, facecolor=color, edgecolor='white', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(i*2+0.75, 0.5, stage, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if i < len(stages)-1:
        ax1.annotate('', xy=(i*2+2, 0.5), xytext=(i*2+1.5, 0.5),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax1.text(6.5, -0.4, '不可变 ←─────────────→ 可变', ha='center', fontsize=9, color='#999')
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-0.8, 1.5)
ax1.axis('off')
ax1.set_title('SkillRelease 制品生命周期', fontsize=12, fontweight='bold')

# Right: canonical_execution state machine
ax2 = axes[1]
states = ['pending', 'running', 'succeeded', 'failed', 'cancelled', 'timeout']
state_colors = ['#FDCB6E', '#74B9FF', '#00B894', '#E17055', '#636E72', '#E17055']
positions = [(1, 3), (3, 3), (5, 4), (5, 2), (5, 1), (5, 0)]

for (x, y), state, color in zip(positions, states, state_colors):
    circle = plt.Circle((x, y), 0.6, color=color, alpha=0.8)
    ax2.add_patch(circle)
    ax2.text(x, y, state, ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax2.annotate('', xy=(2.4, 3), xytext=(1.6, 3), arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
for target_y in [4, 2, 1, 0]:
    ax2.annotate('', xy=(4.4, target_y), xytext=(3.6, 3),
                 arrowprops=dict(arrowstyle='->', color='#999', lw=1, connectionstyle='arc3,rad=0.2'))

ax2.set_xlim(0, 6)
ax2.set_ylim(-1, 5.5)
ax2.axis('off')
ax2.set_title('canonical_execution 六态状态机', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 🧠 架构师思考题

如果 MI 有 3 个不同租户（甲方不同、数据隔离），同一个"合同查询" SkillRelease 怎么部署？是每租户一个 skill_id 还是一个 skill_id 三个 DeploymentRevision？

> **思考方向**：skill_id 是逻辑标识（稳定不变），DeploymentRevision 是物理实例（每租户独立）。同一个 SkillRelease 可以有多个 DeploymentRevision，每个绑定不同租户的配置和数据源。

## 今天多理解了什么

1. **SkillRelease 是制品不是 API**：之前理解为"对外接口"，现在理解为"完整制品"
2. **为什么不允许 latest 依赖**：确定性原则——同一 SkillRelease 在任何环境部署行为必须完全一致
3. **前向回滚的意义**：不是"回退代码"，而是"物化新版本指向旧快照"
4. **存在性保护的架构意义**：404 vs 403 不只是安全措辞，是"不暴露 Skill 存在性"的设计决策

## 📖 术语表

| 英文术语 | 音标 | 释义 |
|---|---|---|
| SkillRelease | /skɪl rɪˈliːs/ | 技能发布制品，唯一可部署单元 |
| DeploymentRevision | /dɪˈplɔɪmənt rɪˈvɪʒən/ | 部署修订，某次部署的冻结快照 |
| ReleaseChannel | /rɪˈliːs ˈtʃænl/ | 发布渠道，控制灰度流量 |
| Release Gate | /rɪˈliːs geɪt/ | 发布门禁，质量检查通过条件 |
| effect_policy | /ɪˈfekt ˈpɒləsi/ | 效果策略，read_only/write/access |
| OCI Artifact | /ˌoʊ-si-aɪ ˈɑːrtɪfækt/ | 开放容器倡议制品格式 |
| digest | /ˈdaɪdʒest/ | 内容寻址哈希，精确标识制品版本 |
| HITL | /eɪtʃ-aɪ-ti-el/ | Human-In-The-Loop，人审回路 |
| cutover | /ˈkʌtˌoʊvər/ | 切换，从旧机制迁移到新机制 |

## ✏️ 课堂练习

**Q1**: SkillRelease A 依赖 SkillRelease B@v1.2，B 发布了 v1.3 修复安全漏洞，A 会自动使用 v1.3 吗？

> 不会。依赖锁精确到 digest，A 仍用 v1.2。要使用 v1.3 必须重新构建 A。

**Q2**: 一个 effect_policy=read_only 的 SkillRelease，用户想修改数据怎么办？

> 创建新 SkillRelease（effect_policy=write），经更严格 Release Gate，发布到更受限 Channel。读写分离。

## 课后测试
1. 描述 SkillRelease 从创建到部署的完整生命周期（5个阶段）
2. 为什么 WorkflowSpec binding 是"技术债"？如果现在就做 cutover，最大的风险是什么？
3. 多租户场景下，同一个 SkillRelease 部署给 3 个租户，DeploymentRevision 是 1 个还是 3 个？

## 🔗 明日连接
Day3：**Deployment / DeploymentRevision** — 为什么 Deployment 独立于 Release？